# NSS 80th Round – Uttarakhand Health Analytics

**Scope:** Uttarakhand (`state = '05'` in level_4/level_5, `state_rc = 4` in level_1a/level_2)  
**Weight variable:** `w1` (frequency weight)

## Output Tables

| # | Topic | Source | Disaggregation |
|---|-------|--------|----------------|
| 1 | IPD Hospitalisation Utilisation (excl. CB 87/88) | level_4 | Rural/Urban · Male/Female · Social Category |
| 2 | Childbirth (CB) Hospitalisation (codes 87/88) | level_4 | Rural/Urban · Male/Female · Social Category |
| 3 | OOPE – IPD Hospitalisation (excl. CB) | level_4 | Rural/Urban · Male/Female · Social Category |
| 4 | OOPE – CB Hospitalisation (87/88) | level_4 | Rural/Urban · Male/Female · Social Category |
| 5 | OPD Utilisation – Ambulatory Care | level_5 | Rural/Urban · Male/Female · Social Category |
| 6 | OOPE – OPD Ambulatory Expenditure | level_5 | Rural/Urban · Male/Female · Social Category |
| 7 | Top-5 Diseases – IPD Burden | level_4 | Rural/Urban · Male/Female · Social Category |
| 8 | Top-5 Diseases – OPD Burden | level_5 | Rural/Urban · Male/Female · Social Category |

**Social Category coding (social_group in level_1a):** 1=ST · 2=SC · 3=OBC · 9=Others/General

## 0. Setup & Paths

In [ ]:
# !pip install pandas numpy -q

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── File paths – adjust if needed ────────────────────────────────────────────
PATH_L1A = 'level_1a.dta'
PATH_L2  = 'level_2.dta'
PATH_L4  = 'level_4.dta'
PATH_L5  = 'level_5.dta'

# ── Uttarakhand codes ─────────────────────────────────────────────────────────
UTTA_STATE_RC = 4          # used in level_1a, level_2
UTTA_STATE    = '05'       # used in level_4, level_5 (string)
CB_CODES      = [87, 88]   # childbirth ailment codes

# ── Social category labels ────────────────────────────────────────────────────
SOC_LABELS = {1: 'ST', 2: 'SC', 3: 'OBC', 9: 'Others/General'}

# ── Ailment name map (NSS 80th round codes) ───────────────────────────────────
AILMENT_NAMES = {
    1: 'Cholera/Diarrhoea',
    2: 'Typhoid/Enteric Fever',
    3: 'Tuberculosis',
    4: 'Malaria/Fever of Unknown Origin',
    5: 'Other Infectious & Parasitic',
    6: 'Cancer/Malignant Neoplasm',
    7: 'Other Neoplasm',
    8: 'Diabetes',
    9: 'Anaemia',
   10: 'Other Endocrine/Metabolic',
   11: 'Mental/Behavioural Disorders',
   12: 'Eye Diseases',
   13: 'Ear Diseases',
   14: 'Hypertension',
   15: 'Heart Diseases (non-hypertensive)',
   16: 'Other Circulatory System Diseases',
   17: 'Pneumonia/Acute Respiratory',
   18: 'Chronic Obstructive Pulmonary',
   19: 'Other Respiratory System',
   20: 'Dental Diseases',
   21: 'Peptic Ulcer',
   22: 'Appendicitis',
   23: 'Hernia',
   24: 'Other Digestive System',
   25: 'Diseases of the Liver/Gallbladder',
   26: 'Kidney/Urinary Tract Diseases',
   27: 'Male Genital Diseases',
   28: 'Gynaecological Diseases',
   29: 'Skin Diseases',
   30: 'Bone/Joint Diseases (non-injury)',
   31: 'Other Musculoskeletal',
   32: 'Congenital Anomalies',
   33: 'Neonatal/Perinatal Conditions',
   34: 'Other Conditions Originating Perinatally',
   35: 'Fever (not classified elsewhere)',
   36: 'Injuries – Road Traffic Accidents',
   37: 'Injuries – Other Accidents',
   38: 'Poisoning',
   39: 'Other Injuries/External Causes',
   40: 'Diseases of Nervous System',
   41: 'Other Diseases',
   42: 'Cataract',
   43: 'Other Eye Conditions',
   44: 'Hydrocele',
   45: 'Haemorrhoids/Piles',
   46: 'Prostate Conditions',
   47: 'Varicose Veins',
   48: 'Tonsils/Adenoids',
   49: 'Gallstones',
   50: 'Kidney Stones',
   51: 'Fractures',
   52: 'Deliveries/Maternity Related',
   53: 'Sterilisation',
   54: 'Other Surgical Conditions',
   87: 'Childbirth – Normal Delivery',
   88: 'Childbirth – Caesarean/Complicated',
   99: 'Other/Not Specified',
}

print('Setup complete.')

## 1. Load All Four Files

In [ ]:
df1a = pd.read_stata(PATH_L1A, convert_categoricals=False)
df2  = pd.read_stata(PATH_L2,  convert_categoricals=False)
df4  = pd.read_stata(PATH_L4,  convert_categoricals=False)
df5  = pd.read_stata(PATH_L5,  convert_categoricals=False)

for nm, d in [('level_1a', df1a), ('level_2', df2), ('level_4', df4), ('level_5', df5)]:
    print(f'{nm:10s}: {d.shape[0]:>7,} rows × {d.shape[1]:>3} cols')

## 2. Build Household Lookup (level_1a) — MPCE + Social Category

In [ ]:
# Numeric conversions for level_1a
for col in ['state_rc', 'mpce_cl', 'social_group']:
    if col in df1a.columns:
        df1a[col] = pd.to_numeric(df1a[col], errors='coerce')

# If column is named differently, try common alternatives
soc_col = None
for candidate in ['social_group', 'soc_grp', 'socgrp', 'religion_caste', 'sc_st']:
    if candidate in df1a.columns:
        soc_col = candidate
        break
if soc_col is None:
    print('WARNING: Social category column not found. Available cols:', df1a.columns.tolist())
else:
    print(f'Social category column: "{soc_col}"')
    print('Value counts:', df1a[soc_col].value_counts().to_dict())

keep_cols = ['hhid', 'state_rc', 'sector', 'mpce_cl']
if soc_col:
    keep_cols.append(soc_col)

hh_lookup = df1a[keep_cols].drop_duplicates('hhid').copy()
if soc_col and soc_col != 'social_group':
    hh_lookup = hh_lookup.rename(columns={soc_col: 'social_group'})
elif soc_col is None:
    hh_lookup['social_group'] = np.nan

hh_lookup['sector']       = pd.to_numeric(hh_lookup['sector'],       errors='coerce')
hh_lookup['mpce_cl']      = pd.to_numeric(hh_lookup['mpce_cl'],      errors='coerce')
hh_lookup['social_group'] = pd.to_numeric(hh_lookup['social_group'], errors='coerce')

print(f'Household lookup: {len(hh_lookup):,} unique HHs')

## 3. Build Person Lookup (level_2) — Sex

In [ ]:
df2['w1']  = pd.to_numeric(df2['w1'],  errors='coerce')
df2['sex'] = df2['sex'].astype(str).str.strip()

# Build indvid = hhid + personslno (zero-padded to 2 digits)
if 'indvid' not in df2.columns:
    df2['indvid'] = df2['hhid'].astype(str) + df2['personslno'].astype(str).str.zfill(2)

person_lookup = df2[['indvid', 'hhid', 'sex']].drop_duplicates('indvid').copy()
print(f'Person lookup: {len(person_lookup):,} individuals')
print('Sex distribution:', df2['sex'].value_counts().to_dict())

## 4. Population Denominators (Uttarakhand, from level_2)

In [ ]:
uk_hhids = df1a[df1a['state_rc'] == UTTA_STATE_RC]['hhid'].unique()
df2_uk   = df2[df2['hhid'].isin(uk_hhids)].copy()

# Merge sector and social_group onto df2_uk
df2_uk = df2_uk.merge(hh_lookup[['hhid','sector','social_group','mpce_cl']], on='hhid', how='left')
df2_uk['sec']    = pd.to_numeric(df2_uk['sector'], errors='coerce')
df2_uk['mpce_cl']= pd.to_numeric(df2_uk['mpce_cl'], errors='coerce')

pop = {
    'total' : df2_uk['w1'].sum(),
    'rural' : df2_uk[df2_uk['sec']==1]['w1'].sum(),
    'urban' : df2_uk[df2_uk['sec']==2]['w1'].sum(),
    'male'  : df2_uk[df2_uk['sex']=='1']['w1'].sum(),
    'female': df2_uk[df2_uk['sex']=='2']['w1'].sum(),
    'ST'    : df2_uk[df2_uk['social_group']==1]['w1'].sum(),
    'SC'    : df2_uk[df2_uk['social_group']==2]['w1'].sum(),
    'OBC'   : df2_uk[df2_uk['social_group']==3]['w1'].sum(),
    'Gen'   : df2_uk[df2_uk['social_group']==9]['w1'].sum(),
}
for q in range(1,6):
    pop[f'q{q}'] = df2_uk[df2_uk['mpce_cl']==q]['w1'].sum()

print('Population denominators (Uttarakhand, weighted):')
for k, v in pop.items():
    print(f'  {k:8s}: {v:>14,.0f}')

## 5. Prepare IPD Dataset (level_4 — Uttarakhand)

In [ ]:
# ── Numeric conversions ───────────────────────────────────────────────────────
for col in ['w1','ailment','sector','ward','free_medical','total_exp','insurance_reim',
            'doctor','medicine','test','bed','other_med','package','transport','other_nonmed']:
    if col in df4.columns:
        df4[col] = pd.to_numeric(df4[col], errors='coerce')

# ── Build indvid for level_4 if not present ───────────────────────────────────
if 'indvid' not in df4.columns:
    df4['indvid'] = df4['hhid'].astype(str) + df4['personslno'].astype(str).str.zfill(2)

# ── Merge person + HH attributes ─────────────────────────────────────────────
df4b = df4.drop(columns=['sector'], errors='ignore')
df4b = df4b.merge(hh_lookup, on='hhid', how='left')
df4b = df4b.merge(person_lookup[['indvid','sex']], on='indvid', how='left')

# state filter — use 'state' column (string '05') if state_rc not in level_4
if 'state_rc' in df4b.columns:
    df4b['state_rc'] = pd.to_numeric(df4b['state_rc'], errors='coerce')
    ipd_uk_all = df4b[df4b['state_rc'] == UTTA_STATE_RC].copy()
else:
    ipd_uk_all = df4b[df4b['state'].astype(str).str.zfill(2) == UTTA_STATE].copy()

ipd_uk_all['oop'] = ipd_uk_all['total_exp'].fillna(0) - ipd_uk_all['insurance_reim'].fillna(0)
ipd_uk_all['ailment_num'] = pd.to_numeric(ipd_uk_all['ailment'], errors='coerce')

# Split: non-CB and CB
ipd_uk     = ipd_uk_all[~ipd_uk_all['ailment_num'].isin(CB_CODES)].copy()   # Tables 1,3,7
ipd_uk_cb  = ipd_uk_all[ ipd_uk_all['ailment_num'].isin(CB_CODES)].copy()   # Tables 2,4

print(f'IPD Uttarakhand (excl. CB): {len(ipd_uk):,} rows')
print(f'IPD Uttarakhand (CB only) : {len(ipd_uk_cb):,} rows')

## 6. Prepare OPD Dataset (level_5 — Uttarakhand)

In [ ]:
# ── Numeric conversions ───────────────────────────────────────────────────────
for col in ['w1','ailment','sector','place_treat','med_institute','free_medical',
            'total_exp','reimburse','doctor','medicine','test','other_med',
            'transport','other_exp','total_medical']:
    if col in df5.columns:
        df5[col] = pd.to_numeric(df5[col], errors='coerce')

# ── Build indvid for level_5 ──────────────────────────────────────────────────
if 'indvid' not in df5.columns:
    df5['indvid'] = df5['hhid'].astype(str) + df5['personslno'].astype(str).str.zfill(2)

# ── Merge HH + person attributes ─────────────────────────────────────────────
df5b = df5.drop(columns=['sector'], errors='ignore')
df5b = df5b.merge(hh_lookup, on='hhid', how='left')
df5b = df5b.merge(person_lookup[['indvid','sex']], on='indvid', how='left')

# State filter
opd_uk = df5b[df5b['state'].astype(str).str.zfill(2) == UTTA_STATE].copy()

# OOP for OPD = total_exp - reimburse
opd_uk['oop']         = opd_uk['total_exp'].fillna(0) - opd_uk['reimburse'].fillna(0)
opd_uk['ailment_num'] = pd.to_numeric(opd_uk['ailment'], errors='coerce')

# Public indicator: place_treat==1 or med_institute==1 → public
if 'place_treat' in opd_uk.columns:
    opd_uk['is_public'] = (opd_uk['place_treat'] == 1).astype(int)
else:
    opd_uk['is_public'] = (opd_uk['med_institute'] == 1).astype(int)

print(f'OPD Uttarakhand: {len(opd_uk):,} rows')
print('Sector split:', opd_uk['sector'].value_counts().to_dict())

## 7. Helper Functions

All metrics are **w1-weighted**. Every function accepts a dataframe subset and returns a scalar.

In [ ]:
def w_sum(df, col='w1'):
    return df[col].sum()

def hosp_rate(sub, pop_val):
    """Hospitalisation / visit rate per 1000 population."""
    if pop_val <= 0:
        return np.nan
    return round(1000 * sub['w1'].sum() / pop_val, 1)

def pub_pct_ipd(sub):
    """% in public facility; ward==1 is govt (level_4)."""
    w = sub['w1'].sum()
    if w <= 0: return np.nan
    return round(100 * sub[sub['ward']==1]['w1'].sum() / w, 1)

def pub_pct_opd(sub):
    """% in public facility (level_5, is_public flag)."""
    w = sub['w1'].sum()
    if w <= 0: return np.nan
    return round(100 * sub[sub['is_public']==1]['w1'].sum() / w, 1)

def ins_pct(sub):
    """% with govt-funded insurance (free_medical==1)."""
    w = sub['w1'].sum()
    if w <= 0: return np.nan
    return round(100 * sub[sub['free_medical']==1]['w1'].sum() / w, 1)

def oop_mean(sub):
    """Weighted mean OOP (INR)."""
    w = sub['w1'].sum()
    if w <= 0: return np.nan
    return round((sub['oop'] * sub['w1']).sum() / w, 0)

def oop_components(sub, medical_cols, nonmed_cols):
    """Returns dict of gross/reimb/net OOP by component."""
    w = sub['w1'].sum()
    if w <= 0:
        return {}
    gross_total  = (sub['total_exp'].fillna(0) * sub['w1']).sum() / w
    reim_total   = (sub[sub.columns.intersection(['insurance_reim','reimburse'])]
                    .iloc[:,0].fillna(0) * sub['w1']).sum() / w
    oop_total    = (sub['oop'] * sub['w1']).sum() / w

    gmed  = sum((sub[c].fillna(0) * sub['w1']).sum() for c in medical_cols if c in sub.columns) / w
    gnmed = sum((sub[c].fillna(0) * sub['w1']).sum() for c in nonmed_cols  if c in sub.columns) / w
    gall  = gmed + gnmed

    rmed  = reim_total * (gmed  / gall) if gall > 0 else np.nan
    rnmed = reim_total * (gnmed / gall) if gall > 0 else np.nan

    return {
        'Gross Expenditure'            : round(gross_total, 0),
        'Insurance Reimbursement'      : round(reim_total,  0),
        'Net OOP'                      : round(oop_total,   0),
        'Gross Medical'                : round(gmed,  0),
        'Gross Non-Medical'            : round(gnmed, 0),
        'Reimb Medical'                : round(rmed,  0) if isinstance(rmed,  float) and not np.isnan(rmed)  else np.nan,
        'Reimb Non-Medical'            : round(rnmed, 0) if isinstance(rnmed, float) and not np.isnan(rnmed) else np.nan,
        'Net Medical'                  : round(gmed  - rmed,  0) if gall > 0 else np.nan,
        'Net Non-Medical'              : round(gnmed - rnmed, 0) if gall > 0 else np.nan,
    }

# ── Disaggregation helper ─────────────────────────────────────────────────────
def disagg_subsets(df):
    """
    Returns ordered dict of (label → subset_df, pop_key).
    Covers: Combined, Rural, Urban, Male, Female, ST, SC, OBC, General.
    """
    subs = {
        'Combined'       : (df,                                         'total'),
        'Rural'          : (df[df['sector']==1],                        'rural'),
        'Urban'          : (df[df['sector']==2],                        'urban'),
        'Male'           : (df[df['sex']=='1'],                         'male'),
        'Female'         : (df[df['sex']=='2'],                         'female'),
        'ST'             : (df[df['social_group']==1],                  'ST'),
        'SC'             : (df[df['social_group']==2],                  'SC'),
        'OBC'            : (df[df['social_group']==3],                  'OBC'),
        'Others/General' : (df[df['social_group']==9],                  'Gen'),
    }
    return subs

# ── HTML table renderer ───────────────────────────────────────────────────────
CSS = '''
<style>
table.nss{border-collapse:collapse;font-family:Arial,sans-serif;font-size:12px;width:100%}
table.nss th{background:#2E75B6;color:#fff;padding:6px 10px;text-align:center}
table.nss th.left{text-align:left}
table.nss td{padding:5px 10px;border-bottom:1px solid #ddd;text-align:right}
table.nss td.row-hdr{text-align:left;font-weight:bold}
table.nss td.section{background:#EBF2FA;text-align:left;font-weight:bold;font-size:11px}
table.nss tr:hover td{background:#f0f5ff}
</style>
'''

DISAGG_COLS = ['Combined','Rural','Urban','Male','Female','ST','SC','OBC','Others/General']

def fmt_val(v, is_inr=False, is_pct=False, is_rate=False):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '—'
    if is_inr:  return f'₹{int(v):,}'
    if is_pct:  return f'{v:.1f}%'
    if is_rate: return f'{v:.1f}'
    return str(v)

def render_table(title, subtitle, rows, col_labels=None):
    """
    rows: list of dicts with keys 'label', 'values' (list aligned to col_labels),
          optional 'is_section' bool, 'is_inr', 'is_pct', 'is_rate'.
    """
    cols = col_labels or DISAGG_COLS
    hdr  = ''.join(f'<th>{c}</th>' for c in cols)
    body = ''
    for r in rows:
        if r.get('is_section'):
            body += f'<tr><td class="section" colspan="{len(cols)+1}">{r["label"]}</td></tr>\n'
            continue
        cells = ''.join(
            f'<td>{fmt_val(v, r.get("is_inr"), r.get("is_pct"), r.get("is_rate"))}</td>'
            for v in r['values']
        )
        body += f'<tr><td class="row-hdr">{r["label"]}</td>{cells}</tr>\n'

    html = f'''{CSS}
    <h3 style="font-family:Arial;margin-bottom:2px">{title}</h3>
    <p style="font-family:Arial;font-size:11px;color:#555;margin-top:0">{subtitle}</p>
    <table class="nss">
      <thead><tr><th class="left">Category</th>{hdr}</tr></thead>
      <tbody>{body}</tbody>
    </table><br>
    '''
    display(HTML(html))

print('Helper functions ready.')

---
## TABLE 1 — IPD Hospitalisation Utilisation (Excl. Childbirth 87/88)

Metrics: Hospitalisation rate per 1000 · % public facility · % govt insurance · Mean OOP (INR)

In [ ]:
subs = disagg_subsets(ipd_uk)

rows_t1 = [
    {'label': 'Hospitalisation Rate (per 1000)', 'is_rate': True,
     'values': [hosp_rate(s, pop[pk]) for s, pk in subs.values()]},
    {'label': '% in Public Facility',            'is_pct':  True,
     'values': [pub_pct_ipd(s) for s, _ in subs.values()]},
    {'label': '% Govt-funded Insurance',          'is_pct':  True,
     'values': [ins_pct(s) for s, _ in subs.values()]},
    {'label': 'Mean OOP Expenditure (INR)',        'is_inr':  True,
     'values': [oop_mean(s) for s, _ in subs.values()]},
]

render_table(
    'Table 1 – IPD Hospitalisation Utilisation (Excl. Childbirth)',
    'Uttarakhand | NSS 80th Round | Excl. ailment codes 87 & 88 | w1-weighted',
    rows_t1
)

---
## TABLE 2 — Childbirth (CB) Hospitalisation (Codes 87/88)

In [ ]:
subs_cb = disagg_subsets(ipd_uk_cb)

rows_t2 = [
    {'label': 'CB Hospitalisation Rate (per 1000)', 'is_rate': True,
     'values': [hosp_rate(s, pop[pk]) for s, pk in subs_cb.values()]},
    {'label': '% in Public Facility',               'is_pct':  True,
     'values': [pub_pct_ipd(s) for s, _ in subs_cb.values()]},
    {'label': '% Govt-funded Insurance',             'is_pct':  True,
     'values': [ins_pct(s) for s, _ in subs_cb.values()]},
    {'label': 'Mean OOP Expenditure (INR)',           'is_inr':  True,
     'values': [oop_mean(s) for s, _ in subs_cb.values()]},
]

render_table(
    'Table 2 – Childbirth Hospitalisation (Codes 87 & 88)',
    'Uttarakhand | NSS 80th Round | Ailment codes 87 & 88 only | w1-weighted',
    rows_t2
)

---
## TABLE 3 — OOPE: IPD Hospitalisation Expenditure (Excl. CB)

Components: Doctor · Medicine · Tests · Bed charges · Other medical · Package · Transport · Other non-medical

In [ ]:
MED_COLS_IPD    = ['doctor','medicine','test','bed','other_med','package']
NONMED_COLS_IPD = ['transport','other_nonmed']

subs = disagg_subsets(ipd_uk)
comp_results = {lbl: oop_components(s, MED_COLS_IPD, NONMED_COLS_IPD) for lbl, (s, _) in subs.items()}

oope_metrics = [
    ('Gross Expenditure',       True,  False),
    ('Insurance Reimbursement', True,  False),
    ('Net OOP',                 True,  False),
    ('Gross Medical',           True,  False),
    ('Gross Non-Medical',       True,  False),
    ('Reimb Medical',           True,  False),
    ('Reimb Non-Medical',       True,  False),
    ('Net Medical',             True,  False),
    ('Net Non-Medical',         True,  False),
]

rows_t3 = []
for met, is_inr, is_pct in oope_metrics:
    rows_t3.append({
        'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
        'values': [comp_results[lbl].get(met, np.nan) for lbl in DISAGG_COLS]
    })

render_table(
    'Table 3 – OOPE: IPD Hospitalisation Expenditure (Excl. Childbirth)',
    'Uttarakhand | NSS 80th Round | Excl. codes 87 & 88 | w1-weighted mean (INR)',
    rows_t3
)

### Table 3b — IPD OOPE: Component-level Detail (per case)

In [ ]:
all_cols = MED_COLS_IPD + NONMED_COLS_IPD

def component_detail(sub, cols):
    w = sub['w1'].sum()
    if w <= 0: return {c: np.nan for c in cols}
    return {c: round((sub[c].fillna(0) * sub['w1']).sum() / w, 0)
            for c in cols if c in sub.columns}

subs = disagg_subsets(ipd_uk)
det_results = {lbl: component_detail(s, all_cols) for lbl, (s, _) in subs.items()}

nice_names = {
    'doctor':'Doctor fees','medicine':'Medicines','test':'Diagnostic tests',
    'bed':'Bed charges','other_med':'Other medical','package':'Package charges',
    'transport':'Transport','other_nonmed':'Other non-medical'
}

rows_t3b = []
rows_t3b.append({'label': '— Medical Components —', 'is_section': True})
for c in MED_COLS_IPD:
    rows_t3b.append({'label': nice_names.get(c,c), 'is_inr': True,
                     'values': [det_results[lbl].get(c, np.nan) for lbl in DISAGG_COLS]})
rows_t3b.append({'label': '— Non-Medical Components —', 'is_section': True})
for c in NONMED_COLS_IPD:
    rows_t3b.append({'label': nice_names.get(c,c), 'is_inr': True,
                     'values': [det_results[lbl].get(c, np.nan) for lbl in DISAGG_COLS]})

render_table(
    'Table 3b – IPD OOPE Component Detail (Excl. Childbirth)',
    'Uttarakhand | NSS 80th Round | Mean gross expenditure per component per case (INR)',
    rows_t3b
)

---
## TABLE 4 — OOPE: Childbirth Hospitalisation Expenditure (Codes 87/88)

In [ ]:
subs_cb = disagg_subsets(ipd_uk_cb)
comp_cb = {lbl: oop_components(s, MED_COLS_IPD, NONMED_COLS_IPD) for lbl, (s, _) in subs_cb.items()}

rows_t4 = []
for met, is_inr, is_pct in oope_metrics:
    rows_t4.append({
        'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
        'values': [comp_cb[lbl].get(met, np.nan) for lbl in DISAGG_COLS]
    })

render_table(
    'Table 4 – OOPE: Childbirth Hospitalisation Expenditure (Codes 87 & 88)',
    'Uttarakhand | NSS 80th Round | w1-weighted mean (INR)',
    rows_t4
)

---
## TABLE 5 — OPD (Ambulatory) Utilisation

Source: level_5. Public = place_treat==1 (govt hospital/dispensary).

In [ ]:
subs_opd = disagg_subsets(opd_uk)

rows_t5 = [
    {'label': 'OPD Visit Rate (per 1000)',   'is_rate': True,
     'values': [hosp_rate(s, pop[pk]) for s, pk in subs_opd.values()]},
    {'label': '% in Public Facility',        'is_pct':  True,
     'values': [pub_pct_opd(s) for s, _ in subs_opd.values()]},
    {'label': '% Govt-funded Insurance',      'is_pct':  True,
     'values': [ins_pct(s) for s, _ in subs_opd.values()]},
    {'label': 'Mean OOP Expenditure (INR)',   'is_inr':  True,
     'values': [oop_mean(s) for s, _ in subs_opd.values()]},
]

render_table(
    'Table 5 – OPD (Ambulatory) Utilisation',
    'Uttarakhand | NSS 80th Round | level_5 | w1-weighted',
    rows_t5
)

---
## TABLE 6 — OOPE: OPD Ambulatory Expenditure

In [ ]:
MED_COLS_OPD    = ['doctor','medicine','test','other_med']
NONMED_COLS_OPD = ['transport','other_exp']

# Harmonise reimbursement column name for OPD
opd_uk['insurance_reim'] = opd_uk.get('reimburse', opd_uk.get('insurance_reim', pd.Series(0, index=opd_uk.index)))
opd_uk['insurance_reim'] = opd_uk['insurance_reim'].fillna(0)

subs_opd = disagg_subsets(opd_uk)
comp_opd = {lbl: oop_components(s, MED_COLS_OPD, NONMED_COLS_OPD) for lbl, (s, _) in subs_opd.items()}

rows_t6 = []
for met, is_inr, is_pct in oope_metrics:
    rows_t6.append({
        'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
        'values': [comp_opd[lbl].get(met, np.nan) for lbl in DISAGG_COLS]
    })

render_table(
    'Table 6 – OOPE: OPD Ambulatory Expenditure',
    'Uttarakhand | NSS 80th Round | level_5 | w1-weighted mean (INR)',
    rows_t6
)

# Component detail
det_opd = {lbl: component_detail(s, MED_COLS_OPD + NONMED_COLS_OPD)
           for lbl, (s, _) in subs_opd.items()}

nice_opd = {'doctor':'Doctor/consultation fee','medicine':'Medicines',
            'test':'Diagnostic tests','other_med':'Other medical',
            'transport':'Transport','other_exp':'Other non-medical'}

rows_t6b = []
rows_t6b.append({'label': '— Medical Components —', 'is_section': True})
for c in MED_COLS_OPD:
    rows_t6b.append({'label': nice_opd.get(c,c), 'is_inr': True,
                     'values': [det_opd[lbl].get(c, np.nan) for lbl in DISAGG_COLS]})
rows_t6b.append({'label': '— Non-Medical Components —', 'is_section': True})
for c in NONMED_COLS_OPD:
    rows_t6b.append({'label': nice_opd.get(c,c), 'is_inr': True,
                     'values': [det_opd[lbl].get(c, np.nan) for lbl in DISAGG_COLS]})

render_table(
    'Table 6b – OPD OOPE Component Detail',
    'Uttarakhand | NSS 80th Round | Mean gross expenditure per component per visit (INR)',
    rows_t6b
)

---
## TABLE 7 — Top-5 Diseases: IPD Burden

Top 5 identified by weighted case count (excl. CB). For each: hosp rate, public %, insurance %, mean OOP — disaggregated.

In [ ]:
# Identify top-5 IPD ailments by weighted cases
top5_ipd = (
    ipd_uk.groupby('ailment_num')['w1'].sum()
    .reset_index()
    .nlargest(5, 'w1')['ailment_num']
    .tolist()
)
print('Top 5 IPD ailments (by weighted cases):')
for code in top5_ipd:
    name = AILMENT_NAMES.get(int(code), f'Code {int(code)}')
    wt   = ipd_uk[ipd_uk['ailment_num']==code]['w1'].sum()
    print(f'  {int(code):>3} – {name:<45} weighted cases: {wt:>10,.0f}')

In [ ]:
for ail_code in top5_ipd:
    ail_name  = AILMENT_NAMES.get(int(ail_code), f'Code {int(ail_code)}')
    df_ail    = ipd_uk[ipd_uk['ailment_num'] == ail_code]
    subs_ail  = disagg_subsets(df_ail)

    rows_ail = [
        {'label': 'Hosp Rate (per 1000)',     'is_rate': True,
         'values': [hosp_rate(s, pop[pk]) for s, pk in subs_ail.values()]},
        {'label': '% Public Facility',        'is_pct':  True,
         'values': [pub_pct_ipd(s) for s, _ in subs_ail.values()]},
        {'label': '% Govt Insurance',         'is_pct':  True,
         'values': [ins_pct(s) for s, _ in subs_ail.values()]},
        {'label': 'Mean OOP (INR)',            'is_inr':  True,
         'values': [oop_mean(s) for s, _ in subs_ail.values()]},
        {'label': '— OOPE Components —',      'is_section': True},
    ]
    comp_ail = {lbl: oop_components(s, MED_COLS_IPD, NONMED_COLS_IPD)
                for lbl, (s, _) in subs_ail.items()}
    for met, is_inr, is_pct in oope_metrics:
        rows_ail.append({
            'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
            'values': [comp_ail[lbl].get(met, np.nan) for lbl in DISAGG_COLS]
        })

    render_table(
        f'Table 7 – IPD: {ail_name} (Code {int(ail_code)})',
        'Uttarakhand | NSS 80th Round | w1-weighted | Part of Top-5 IPD diseases',
        rows_ail
    )

---
## TABLE 8 — Top-5 Diseases: OPD Burden

Top 5 by weighted OPD visit count (level_5, Uttarakhand).

In [ ]:
top5_opd = (
    opd_uk.groupby('ailment_num')['w1'].sum()
    .reset_index()
    .nlargest(5, 'w1')['ailment_num']
    .tolist()
)
print('Top 5 OPD ailments (by weighted visits):')
for code in top5_opd:
    name = AILMENT_NAMES.get(int(code), f'Code {int(code)}')
    wt   = opd_uk[opd_uk['ailment_num']==code]['w1'].sum()
    print(f'  {int(code):>3} – {name:<45} weighted visits: {wt:>10,.0f}')

In [ ]:
for ail_code in top5_opd:
    ail_name  = AILMENT_NAMES.get(int(ail_code), f'Code {int(ail_code)}')
    df_ail    = opd_uk[opd_uk['ailment_num'] == ail_code]
    subs_ail  = disagg_subsets(df_ail)

    rows_ail = [
        {'label': 'OPD Visit Rate (per 1000)', 'is_rate': True,
         'values': [hosp_rate(s, pop[pk]) for s, pk in subs_ail.values()]},
        {'label': '% Public Facility',         'is_pct':  True,
         'values': [pub_pct_opd(s) for s, _ in subs_ail.values()]},
        {'label': '% Govt Insurance',          'is_pct':  True,
         'values': [ins_pct(s) for s, _ in subs_ail.values()]},
        {'label': 'Mean OOP (INR)',             'is_inr':  True,
         'values': [oop_mean(s) for s, _ in subs_ail.values()]},
        {'label': '— OOPE Components —',       'is_section': True},
    ]
    comp_ail = {lbl: oop_components(s, MED_COLS_OPD, NONMED_COLS_OPD)
                for lbl, (s, _) in subs_ail.items()}
    for met, is_inr, is_pct in oope_metrics:
        rows_ail.append({
            'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
            'values': [comp_ail[lbl].get(met, np.nan) for lbl in DISAGG_COLS]
        })

    render_table(
        f'Table 8 – OPD: {ail_name} (Code {int(ail_code)})',
        'Uttarakhand | NSS 80th Round | w1-weighted | Part of Top-5 OPD diseases',
        rows_ail
    )

---
## Export All Tables to CSV

In [ ]:
def table_to_df(rows, col_labels=None):
    cols = col_labels or DISAGG_COLS
    records = []
    for r in rows:
        if r.get('is_section'):
            continue
        rec = {'Metric': r['label']}
        for c, v in zip(cols, r['values']):
            rec[c] = v
        records.append(rec)
    return pd.DataFrame(records)

# Build export dicts
export_tables = {
    'T1_IPD_Utilisation'     : rows_t1,
    'T2_CB_Utilisation'      : rows_t2,
    'T3_IPD_OOPE'            : rows_t3,
    'T3b_IPD_OOPE_Component' : rows_t3b,
    'T4_CB_OOPE'             : rows_t4,
    'T5_OPD_Utilisation'     : rows_t5,
    'T6_OPD_OOPE'            : rows_t6,
    'T6b_OPD_OOPE_Component' : rows_t6b,
}

for name, rows in export_tables.items():
    fname = f'UK_{name}.csv'
    table_to_df(rows).to_csv(fname, index=False)
    print(f'Saved: {fname}')

# Top-5 exports
for ail_code in top5_ipd:
    ail_name = AILMENT_NAMES.get(int(ail_code), f'Code{int(ail_code)}').replace(' ','_').replace('/','_')
    df_ail   = ipd_uk[ipd_uk['ailment_num'] == ail_code]
    subs_ail = disagg_subsets(df_ail)
    r_rows   = []
    r_rows.append({'label': 'Hosp Rate (per 1000)', 'is_rate': True,
                   'values': [hosp_rate(s, pop[pk]) for s, pk in subs_ail.values()]})
    r_rows.append({'label': '% Public', 'is_pct': True,
                   'values': [pub_pct_ipd(s) for s, _ in subs_ail.values()]})
    r_rows.append({'label': 'Mean OOP', 'is_inr': True,
                   'values': [oop_mean(s) for s, _ in subs_ail.values()]})
    comp_ail = {lbl: oop_components(s, MED_COLS_IPD, NONMED_COLS_IPD)
                for lbl, (s, _) in subs_ail.items()}
    for met, is_inr, is_pct in oope_metrics:
        r_rows.append({'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
                       'values': [comp_ail[lbl].get(met, np.nan) for lbl in DISAGG_COLS]})
    fname = f'UK_T7_IPD_{ail_name[:30]}.csv'
    table_to_df(r_rows).to_csv(fname, index=False)
    print(f'Saved: {fname}')

for ail_code in top5_opd:
    ail_name = AILMENT_NAMES.get(int(ail_code), f'Code{int(ail_code)}').replace(' ','_').replace('/','_')
    df_ail   = opd_uk[opd_uk['ailment_num'] == ail_code]
    subs_ail = disagg_subsets(df_ail)
    r_rows   = []
    r_rows.append({'label': 'OPD Visit Rate (per 1000)', 'is_rate': True,
                   'values': [hosp_rate(s, pop[pk]) for s, pk in subs_ail.values()]})
    r_rows.append({'label': '% Public', 'is_pct': True,
                   'values': [pub_pct_opd(s) for s, _ in subs_ail.values()]})
    r_rows.append({'label': 'Mean OOP', 'is_inr': True,
                   'values': [oop_mean(s) for s, _ in subs_ail.values()]})
    comp_ail = {lbl: oop_components(s, MED_COLS_OPD, NONMED_COLS_OPD)
                for lbl, (s, _) in subs_ail.items()}
    for met, is_inr, is_pct in oope_metrics:
        r_rows.append({'label': met, 'is_inr': is_inr, 'is_pct': is_pct,
                       'values': [comp_ail[lbl].get(met, np.nan) for lbl in DISAGG_COLS]})
    fname = f'UK_T8_OPD_{ail_name[:30]}.csv'
    table_to_df(r_rows).to_csv(fname, index=False)
    print(f'Saved: {fname}')

print('\nAll exports complete.')

---
## Variable Reference

### level_4 (IPD)
| Variable | Meaning |
|---|---|
| `ward` | 1=Govt, 2=Private, 3=Other |
| `free_medical` | 1=Fully free (Govt scheme), 2=Fully free (Employer), 3=Partly free, 4=Not free |
| `ailment` | Disease code (numeric) |
| `total_exp` | Total hospitalisation expenditure (INR) |
| `insurance_reim` | Reimbursement received (INR) |
| `oop` | Net OOP = total_exp − insurance_reim |

### level_5 (OPD)
| Variable | Meaning |
|---|---|
| `place_treat` | 1=Govt hosp/dispensary (public), 2=Private, 4=Charitable, 5=Other |
| `free_medical` | Same coding as level_4 |
| `total_exp` | Total OPD expenditure (INR) |
| `reimburse` | Reimbursement received (INR) |
| `oop` | Net OOP = total_exp − reimburse |

### level_1a (Household)
| Variable | Meaning |
|---|---|
| `social_group` | 1=ST, 2=SC, 3=OBC, 9=Others/General |
| `sector` | 1=Rural, 2=Urban |
| `mpce_cl` | MPCE quintile (1=Poorest … 5=Richest) |
| `state_rc` | State code (Uttarakhand = 4) |

### Disaggregation columns in all tables
`Combined · Rural · Urban · Male · Female · ST · SC · OBC · Others/General`